
- **ROC-AUC** — умеет ли модель упорядочивать пассажиров по риску: это вероятность того, что случайному выжившему модель даст балл выше, чем случайному погибшему. 1.0 — идеал, 0.5 — монетка. Менее 0.5 - что-то сильно не так.
- **Log-loss** — штраф за уверенные ошибки: ошибиться с вероятностью 0.99 намного дороже, чем с 0.51. Меньше — лучше.

Подготовка данных и модели — аналогично предыдущим

In [ ]:
# --- Подготовка данных ---
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, roc_auc_score, log_loss

df = pd.read_csv('Titanic-Dataset.csv')

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['TicketGroupSize'] = df.groupby('Ticket')['Ticket'].transform('count')
df['FarePerPersonLog'] = np.log1p(df['Fare'] / df['TicketGroupSize'])

df_ml = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)
X = df_ml[['Pclass', 'Age', 'FarePerPersonLog', 'FamilySize', 'Sex_male', 'Embarked_Q', 'Embarked_S']]
y = df_ml['Survived']
X = X.fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Модели ---
models = {
    "Dummy (все погибли)": DummyClassifier(strategy="most_frequent"),
    "Linear (Logistic Regression)": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=4, random_state=42, eval_metric='logloss'),
    "CatBoost": CatBoostClassifier(iterations=150, depth=4, random_state=42, verbose=0)
}

# --- Обучение и оценка по вероятностям ---
results = {}
probas = {}  # вероятности класса «выжил» — пригодятся для ROC-кривых
for name, model in models.items():
    # Для логистической регрессии и KNN используем отмасштабированные данные
    if name in ["Linear (Logistic Regression)", "KNN"]:
        model.fit(X_train_scaled, y_train)
        proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
    probas[name] = proba

    preds = (proba >= 0.5).astype(int)  # метка = вероятность против порога 0.5
    results[name] = {
        "F1": f1_score(y_test, preds, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, proba),
        "Log-loss": log_loss(y_test, proba),
    }

# Таблица: сортируем по ROC-AUC, у Log-loss лучше меньшее значение
sorted_results = sorted(results.items(), key=lambda x: x[1]["ROC-AUC"], reverse=True)
df_results = pd.DataFrame([{"Модель": name, **m} for name, m in sorted_results]).round(3)
df_results

,Модель,F1,ROC-AUC,Log-loss
0,CatBoost,0.705,0.848,0.448
1,Random Forest,0.667,0.842,0.454
2,XGBoost,0.754,0.830,0.509
3,KNN,0.718,0.828,2.138
4,Linear (Logistic Regression),0.733,0.826,0.479
5,Decision Tree,0.722,0.793,1.289
6,Dummy (все погибли),0.000,0.500,13.894


ROC-кривая показывает модель сразу на всех порогах: по оси X — доля ложных тревог (FPR), по оси Y — доля найденных выживших (TPR, он же Recall). Чем ближе кривая к левому верхнему углу, тем лучше; диагональ — случайное угадывание.

In [2]:
from sklearn.metrics import roc_curve

fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Случайное угадывание",
                         line=dict(dash="dash", color="gray")))
for name, proba in probas.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines",
                             name=f"{name} (AUC={results[name]['ROC-AUC']:.3f})"))

fig.update_layout(
    template="plotly_white",
    height=550, width=850,
    title_text="ROC-кривые моделей",
    title_x=0.5,
    xaxis_title="FPR — доля ложных тревог",
    yaxis_title="TPR (Recall) — доля найденных выживших",
)
fig.show()

- Настоящие модели лежат заметно выше диагонали (ROC-AUC ~0.85); Dummy — ровно на диагонали (AUC = 0.5)
- Log-loss осмысленна — все модели выдают вероятности. У Dummy она огромна: пустышка со 100% уверенностью говорит «погиб» каждому и получает штраф за каждого выжившего.
- ROC-AUC дополняет F1: F1 оценивает решение при конкретном пороге 0.5, ROC-AUC — качество ранжирования на всех порогах сразу. Если по F1 модели идут плотно, а по ROC-AUC одна впереди — у неё есть запас, который можно добрать подбором порога.